# Data Collection

In [2]:
import pandas as pd
import numpy as np

##########################################
'''DATA'''
##########################################
alpha = 0.3 #TODO Trouver une source qui justifie ce choix 
Tmax= 15 #au dessus : pas de chauffage
Tmin = 0 #en dessous: chauffage maximal
cos_phi = 0.95

##########################################
# 1. Charger data du fichier 87_grid
file = "87_0_grid.xlsx"

parameters = pd.read_excel(file, sheet_name="parameters")
f =  parameters["f_hz"].values[0] 
omega = 2 * np.pi * f

base = pd.read_excel(file, sheet_name="res_ext_grid") #TODO : CHANGER AVEC NOUVEAU OPF POUR SCENARIO 2 
P_base = base["p_mw"].values
Q_base = base["q_mvar"].values

load = pd.read_excel(file, sheet_name="load")
P_load = load.groupby("bus")["p_mw"].sum() #tableau avec pour chaque node une valeur de load associée 
Q_load = load.groupby("bus")["q_mvar"].sum()

bus = pd.read_excel(file, sheet_name="bus")
n_nodes = len(bus)
nodes = range(n_nodes)

bus_ref = pd.read_excel(file, sheet_name="res_bus") #TODO : CHANGER AVEC NOUVEAU OPF POUR SCENARIO 2 
P_ref = bus_ref["p_mw"].values
Q_ref = bus_ref["q_mvar"].values

lines = pd.read_excel(file, sheet_name="line")
#lines["busi - busj"] = lines["from_bus"].astype(str) + "-" + lines["to_bus"].astype(str)
line_data = pd.DataFrame({
    #"id": lines["id"],
    "from_bus": lines["from_bus"],
    "to_bus": lines["to_bus"],
    "length": lines["length_km"],
    "r": lines["r_ohm_per_km"] * lines["length_km"],
    "x": lines["x_ohm_per_km"] * lines["length_km"],
    "c": lines["c_nf_per_km"] * lines["length_km"] * 1e-9,
    "I_max": lines["max_i_ka"]
})

#Construire gij, bij, bij(sh)
line_data["g"] = line_data["r"] / (line_data["r"]**2 + line_data["x"]**2)
line_data["b"] = -line_data["x"] / (line_data["r"]**2 + line_data["x"]**2)
line_data["b_sh"] = omega * line_data["c"]
#print(line_data.head()) 

#Construire les matrices Jacobiennes
# Initialisation des matrices
J_Ptheta = np.zeros((n_nodes, n_nodes))
J_QU     = np.zeros((n_nodes, n_nodes))
J_PU     = np.zeros((n_nodes, n_nodes))
# Boucle sur chaque ligne ij
for l in range(len(line_data)):
    i = int(line_data.loc[l, "from_bus"])
    j = int(line_data.loc[l, "to_bus"])

    gij = line_data.loc[l, "g"]
    bij = line_data.loc[l, "b"]
    bsh = line_data.loc[l, "b_sh"]
# 1. MATRICE J_Ptheta
    # diagonale
    J_Ptheta[i, i] -= bij
    J_Ptheta[j, j] -= bij
    # hors diagonale
    J_Ptheta[i, j] += bij
    J_Ptheta[j, i] += bij
# 2. MATRICE J_QU
    # diagonale
    J_QU[i, i] -= (2*bsh + bij)
    J_QU[j, j] -= (2*bsh + bij)
    # hors diagonale
    J_QU[i, j] += bij
    J_QU[j, i] += bij
# 3. MATRICE J_PU
    # diagonale
    J_PU[i, i] += gij
    J_PU[j, j] += gij
    # hors diagonale
    J_PU[i, j] -= gij
    J_PU[j, i] -= gij

#Calculer S_max 
line_data["S_max"] = np.sqrt(3) * line_data["I_max"] * line_data["from_bus"].apply(lambda i: bus.loc[i, "vn_kv"])

##########################################
# 2. Charger data des PV
irr = pd.read_csv("irradiance_hourly.csv")
G = irr["irradiance_W_m2"].values  # taille 8760
G_norm = G / np.max(G)

pv_data = []    #Data PV par load 
pv_data_dt = [] #Data PV par load par heure de l'année 
for k in range(n_nodes):
    Pk = P_load.get(k, 0) # load au node k
    Ppv = alpha * Pk #Ppv au node k 
    Qpv = Ppv * np.tan(np.arccos(cos_phi))
    pv_data.append({
        "bus": k,
        "P_pv": Ppv,
        "Q_pv": Qpv
    })
    for t in range(len(G)):
        Ppv = alpha * Pk * G_norm[t]
        Qpv = Ppv * np.tan(np.arccos(cos_phi))
        pv_data_dt.append({
            "bus": k,
            "time": t,
            "P_pv": Ppv,
            "Q_pv": Qpv
        })
pv_df = pd.DataFrame(pv_data)
pv_df.to_csv("pv_data.csv", index=False)
pv_df_dt = pd.DataFrame(pv_data_dt)
pv_df_dt.to_csv("pv_data_dt.csv", index=False)


##########################################
# 3. Charger data des HP
COP = 3.9         # heat pump (nPro) #TODO changer quand zone bien délimitée
P_hp_total = 26072 / 1000 # MW thermique (nPro) #TODO changer quand zone bien délimitée
temp = pd.read_csv("temperature_hourly.csv") #TODO changer avec bon fichier

T = temp["temperature_C"].values
f_t = np.clip((Tmax - T) / (Tmax - Tmin), 0, 1) # Fonction chauffage qui indique consommation de l'HP en fonction de la temperature exterieure 

P_hp_total_elec = P_hp_total / COP #puissance electrique totale 
Pk_total = P_load.sum() # Total load

hp_data = []
hp_data_dt = []
for k in range(n_nodes):
    Pk = P_load.get(k, 0)
    P_hp = P_hp_total_elec * (Pk / Pk_total)
    hp_data.append({
        "bus": k,
        "P_hp": P_hp
    })
    for t in range(len(T)):
        P_hp_dt = P_hp * f_t[t]
        hp_data_dt.append({
            "bus": k,
            "time": t,
            "P_hp": P_hp_dt
        })
hp_df = pd.DataFrame(hp_data)
hp_df.to_csv("hp_data.csv", index=False)
hp_df_dt = pd.DataFrame(hp_data_dt)
hp_df_dt.to_csv("hp_data_dt.csv", index=False)


# CREE LE MODELE

In [ ]:
import gurobipy as gp
from gurobipy import GRB

